In [ ]:
# Basic M1-friendly TensorFlow Pipeline (Oral Cavity)

This notebook breaks the pipeline into small, debuggable steps:
- Config
- Data loading (with fallback if you only have a single folder)
- Simple model (small CNN; optional MobileNetV2)
- Train
- Evaluate (accuracy)
- Save
- Reload and test inference

**Note: This version includes 5 classes including "Unknown" for production robustness.**

## Data Structure Expected:
```
your_dataset/
├── train/
│   ├── Benign/
│   ├── Healthy/
│   ├── OCA/
│   ├── OPMD/
│   └── Unknown/          # Non-oral images, poor quality, etc.
├── val/
│   └── [same structure]
└── test/
    └── [same structure]
```

**Unknown class should contain:**
- Non-oral cavity images (hands, faces, objects, etc.)
- Poor quality oral images (blurry, too dark, etc.)
- Images that don't clearly show oral cavity

Tip: Start with very small `batch_size` and `epochs` on M1 to validate the flow first.


In [12]:
import os, json, random, pathlib
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(tf.__version__)

# M1-friendly defaults
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Avoid mixed precision on M1 CPU/GPU
# tf.keras.mixed_precision.set_global_policy('float32')

ORAL_DATA_DIR_DEFAULT = "~/dev/oral_cavity_dataset_clean"
config = {
    "data_dir": os.environ.get("ORAL_DATA_DIR", ORAL_DATA_DIR_DEFAULT),
    "image_size": (224, 224),
    "batch_size": 8,
    "epochs": 2,
    "class_names": ["Benign", "Healthy", "OCA", "OPMD", "Unknown"],
    "model_name": "basic_cnn",
    "output_root": "./ml/training/results/basic_m1",
}

config["data_dir"] = os.path.expanduser(config["data_dir"]) 
pathlib.Path(config["output_root"]).mkdir(parents=True, exist_ok=True)

with open(os.path.join(config["output_root"], "config_used.json"), "w") as f:
    json.dump(config, f, indent=2)

config


2.15.0


{'data_dir': '/Users/alokdwivedi/dev/oral_cavity_dataset_clean',
 'image_size': (224, 224),
 'batch_size': 8,
 'epochs': 2,
 'class_names': ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown'],
 'model_name': 'basic_cnn',
 'output_root': './ml/training/results/basic_m1'}

In [14]:
from pathlib import Path
from typing import Tuple
import cv2
import os

AUTOTUNE = tf.data.AUTOTUNE

def get_split_dirs(data_dir: str):
    base = Path(data_dir)
    train_dir = base / "train"
    val_dir = base / "val"
    test_dir = base / "test"
    return train_dir, val_dir, test_dir


def has_split_structure(data_dir: str) -> bool:
    train_dir, val_dir, test_dir = get_split_dirs(data_dir)
    return train_dir.exists() and val_dir.exists() and test_dir.exists()


def validate_images_in_directory(directory: Path) -> list:
    """Check for corrupted images and return list of bad files"""
    bad_files = []
    print(f"🔍 Validating images in {directory}...")
    
    for class_dir in directory.iterdir():
        if class_dir.is_dir():
            print(f"  Checking {class_dir.name}...")
            # Fix: Convert generators to lists before concatenating
            img_files = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.jpeg")) + list(class_dir.glob("*.png"))
            for img_file in img_files:
                try:
                    # Try to read with OpenCV
                    img = cv2.imread(str(img_file))
                    if img is None:
                        bad_files.append(str(img_file))
                        print(f"    ❌ {img_file.name} - OpenCV can't read")
                        continue
                    
                    # Try to decode with TensorFlow
                    img_bytes = tf.io.read_file(str(img_file))
                    tf.image.decode_image(img_bytes)
                    
                except Exception as e:
                    bad_files.append(str(img_file))
                    print(f"    ❌ {img_file.name} - {str(e)[:50]}...")
    
    if bad_files:
        print(f"⚠️  Found {len(bad_files)} corrupted images!")
        print("💡 Consider removing or fixing these files before training.")
    else:
        print("✅ All images validated successfully!")
    
    return bad_files


def build_tfds_from_directory(directory: Path, image_size: Tuple[int, int], batch_size: int, shuffle: bool) -> tuple:
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="int",
        image_size=image_size,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=SEED,
    )
    # Get class names before applying transformations
    class_names = ds.class_names
    
    # Normalize to [0,1]
    def normalize(img, label):
        img = tf.cast(img, tf.float32) / 255.0
        return img, label
    ds = ds.map(normalize, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE), class_names


# Validate data first
if has_split_structure(config["data_dir"]):
    train_dir, val_dir, test_dir = get_split_dirs(config["data_dir"]) 
    print("Using train/val/test split directories")
    
    # Validate all directories
    bad_files = []
    # Skipping for now as ist already done once - NOW we dont have bad images
    # bad_files.extend(validate_images_in_directory(train_dir))
    # bad_files.extend(validate_images_in_directory(val_dir))
    # bad_files.extend(validate_images_in_directory(test_dir))
    
    if bad_files:
        print(f"\n⚠️  Found {len(bad_files)} corrupted files. Please fix or remove them.")
        print("You can remove them with:")
        for bad_file in bad_files[:5]:  # Show first 5
            print(f"  rm '{bad_file}'")
        if len(bad_files) > 5:
            print(f"  ... and {len(bad_files) - 5} more")
    else:
        print("✅ Data validation passed!")
        train_ds, actual_class_names = build_tfds_from_directory(train_dir, config["image_size"], config["batch_size"], shuffle=True)
        val_ds, _   = build_tfds_from_directory(val_dir,   config["image_size"], config["batch_size"], shuffle=False)
        test_ds, _  = build_tfds_from_directory(test_dir,  config["image_size"], config["batch_size"], shuffle=False)
        
        # Get the actual class names from the dataset (alphabetically sorted)
        print(f"\n🔍 Actual class mapping (from dataset):")
        for i, class_name in enumerate(actual_class_names):
            print(f"  Index {i}: {class_name}")
        
        # Update config with actual class names
        config["class_names"] = actual_class_names
        print(f"\n✅ Updated config['class_names'] to match dataset order")
        
else:
    # Fallback: single folder with subfolders per class -> split on the fly
    print("No split found; creating a simple 80/10/10 split from root directory")
    root = Path(config["data_dir"]) 
    
    # Validate root directory
    bad_files = validate_images_in_directory(root)
    
    if bad_files:
        print(f"\n⚠️  Found {len(bad_files)} corrupted files. Please fix or remove them.")
    else:
        print("✅ Data validation passed!")
        full_ds = tf.keras.preprocessing.image_dataset_from_directory(
            root, labels="inferred", label_mode="int", image_size=config["image_size"],
            batch_size=config["batch_size"], shuffle=True, seed=SEED)
        actual_class_names = full_ds.class_names
        print("Classes:", actual_class_names)
        
        # Get the actual class names from the dataset (alphabetically sorted)
        print(f"\n🔍 Actual class mapping (from dataset):")
        for i, class_name in enumerate(actual_class_names):
            print(f"  Index {i}: {class_name}")
        
        # Update config with actual class names
        config["class_names"] = actual_class_names
        print(f"\n✅ Updated config['class_names'] to match dataset order")
        val_size = int(0.1 * tf.data.experimental.cardinality(full_ds).numpy())
        test_size = val_size
        train_size = tf.data.experimental.cardinality(full_ds).numpy() - val_size - test_size
        train_ds = full_ds.take(train_size)
        tmp = full_ds.skip(train_size)
        val_ds = tmp.take(val_size)
        test_ds = tmp.skip(val_size)

# Only create datasets if validation passed
if 'train_ds' in locals():
    print(f"✅ Datasets created successfully!")
    print(f"Train batches: {tf.data.experimental.cardinality(train_ds)}")
    print(f"Val batches: {tf.data.experimental.cardinality(val_ds)}")
    print(f"Test batches: {tf.data.experimental.cardinality(test_ds)}")
else:
    print("❌ Please fix corrupted images before proceeding to model training.")


Using train/val/test split directories
✅ Data validation passed!
Found 2757 files belonging to 5 classes.
Found 531 files belonging to 5 classes.
Found 534 files belonging to 5 classes.

🔍 Actual class mapping (from dataset):
  Index 0: Benign
  Index 1: Healthy
  Index 2: OCA
  Index 3: OPMD
  Index 4: Unknown

✅ Updated config['class_names'] to match dataset order
✅ Datasets created successfully!
Train batches: 345
Val batches: 67
Test batches: 67


In [22]:
# Calculate class weights for imbalanced dataset
from sklearn.utils.class_weight import compute_class_weight

def calculate_class_weights(data_dir: str, class_names: list) -> dict:
    """Calculate class weights for handling imbalanced dataset"""
    train_path = Path(data_dir) / 'train'
    class_counts = []
    
    print("📊 Analyzing class distribution...")
    
    for class_name in class_names:
        class_path = train_path / class_name
        if class_path.exists():
            count = len(list(class_path.glob('*.jpg')) + 
                       list(class_path.glob('*.jpeg')) + 
                       list(class_path.glob('*.png')))
            class_counts.append(count)
        else:
            class_counts.append(0)
    
    # Calculate weights using sklearn
    labels = []
    for idx, count in enumerate(class_counts):
        labels.extend([idx] * count)
    
    if len(labels) == 0:
        print("⚠️ No training data found!")
        return {}
    
    class_weights_array = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(labels),
        y=labels
    )
    
    class_weights = {i: weight for i, weight in enumerate(class_weights_array)}
    
    print("\n📈 Class distribution and weights:")
    total_samples = sum(class_counts)
    for i, (class_name, count) in enumerate(zip(class_names, class_counts)):
        percentage = (count / total_samples) * 100 if total_samples > 0 else 0
        print(f"  {class_name:8}: {count:4} samples ({percentage:5.1f}%) -> weight: {class_weights[i]:.3f}")
    
    # Check for severe imbalance
    max_count = max(class_counts) if class_counts else 1
    min_count = min(class_counts) if class_counts else 1
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    
    if imbalance_ratio > 10:
        print(f"⚠️  Severe class imbalance detected (ratio: {imbalance_ratio:.1f}:1)")
        print("💡 Consider collecting more samples for minority classes")
    elif imbalance_ratio > 3:
        print(f"⚠️  Moderate class imbalance detected (ratio: {imbalance_ratio:.1f}:1)")
        print("💡 Class weights will help balance training")
    else:
        print("✅ Classes are reasonably balanced")
    
    return class_weights

# Calculate class weights
class_weights = calculate_class_weights(config["data_dir"], config["class_names"])


📊 Analyzing class distribution...

📈 Class distribution and weights:
  Benign  :  656 samples ( 23.8%) -> weight: 0.841
  Healthy :  510 samples ( 18.5%) -> weight: 1.081
  OCA     :  216 samples (  7.8%) -> weight: 2.553
  OPMD    :  975 samples ( 35.4%) -> weight: 0.566
  Unknown :  400 samples ( 14.5%) -> weight: 1.379
⚠️  Moderate class imbalance detected (ratio: 4.5:1)
💡 Class weights will help balance training


In [23]:
def build_basic_cnn(input_shape=(224,224,3), num_classes=5):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1./1.0)(inputs)  # no-op placeholder

    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = keras.Model(inputs, outputs)
    return model

num_classes = len(config["class_names"]) if has_split_structure(config["data_dir"]) else None
if num_classes is None:
    # Attempt to infer from dataset object created above if fallback path
    try:
        num_classes = train_ds.element_spec[1].shape[0]
    except Exception:
        num_classes = 5  # Updated for 5 classes including Unknown

model = build_basic_cnn((*config["image_size"], 3), num_classes)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)
model.summary()


Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 rescaling_3 (Rescaling)     (None, 224, 224, 3)       0         
                                                                 
 conv2d_9 (Conv2D)           (None, 224, 224, 32)      896       
                                                                 
 batch_normalization_9 (Bat  (None, 224, 224, 32)      128       
 chNormalization)                                                
                                                                 
 re_lu_9 (ReLU)              (None, 224, 224, 32)      0         
                                                                 
 max_pooling2d_9 (MaxPoolin  (None, 112, 112, 32)      0         
 g2D)                                                      

In [24]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
]

steps_per_epoch = None
validation_steps = None

# Use class weights if available
fit_kwargs = {
    'epochs': config["epochs"],
    'steps_per_epoch': steps_per_epoch,
    'validation_steps': validation_steps,
    'verbose': 1,
    'callbacks': callbacks,
}

if class_weights:
    fit_kwargs['class_weight'] = class_weights
    print("🎯 Using class weights for imbalanced dataset")
else:
    print("⚠️ No class weights available - training without balancing")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    **fit_kwargs
)

import pandas as pd
hist_df = pd.DataFrame(history.history)
hist_path = os.path.join(config["output_root"], "training_history.csv")
hist_df.to_csv(hist_path, index=False)
print("Saved:", hist_path)
hist_df.tail()


🎯 Using class weights for imbalanced dataset
Epoch 1/2


2025-09-02 17:50:48.215163: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


345/345 [==============================] - 31s 85ms/step - loss: 1.3036 - accuracy: 0.4001 - val_loss: 1.9921 - val_accuracy: 0.0697
Epoch 2/2
345/345 [==============================] - 30s 86ms/step - loss: 1.1581 - accuracy: 0.4331 - val_loss: 1.2454 - val_accuracy: 0.3879
Saved: ./ml/training/results/basic_m1/training_history.csv


,loss,accuracy,val_loss,val_accuracy
0,1.303583,0.400073,1.992105,0.069680
1,1.158090,0.433079,1.245445,0.387947


In [25]:
eval_results = model.evaluate(test_ds, verbose=1)
print(dict(zip(model.metrics_names, eval_results)))


67/67 [==============================] - 5s 69ms/step - loss: 1.2733 - accuracy: 0.3933
{'loss': 1.2733498811721802, 'accuracy': 0.3932584226131439}


In [26]:
save_dir = os.path.join(config["output_root"], "model")
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, "basic_model.h5")
model.save(model_path)
print("Model saved to:", model_path)


Model saved to: ./ml/training/results/basic_m1/model/basic_model.h5


/Users/alokdwivedi/dev/github/medpylabs/oral_cavity_classifier_app/ml/training/venv_oral_cancer_classifier/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [27]:
# Convert to TFLite
print("🔄 Converting model to TFLite...")

# Create TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Set optimizations for smaller model size
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Convert the model
try:
    tflite_model = converter.convert()
    
    # Save TFLite model
    tflite_path = os.path.join(save_dir, "basic_model.tflite")
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    
    print(f"✅ TFLite model saved to: {tflite_path}")
    print(f"📦 Model size: {len(tflite_model) / (1024*1024):.2f} MB")
    
except Exception as e:
    print(f"❌ TFLite conversion failed: {e}")
    tflite_path = None


🔄 Converting model to TFLite...
INFO:tensorflow:Assets written to: /var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/tmp8jjt7rky/assets


INFO:tensorflow:Assets written to: /var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/tmp8jjt7rky/assets


✅ TFLite model saved to: ./ml/training/results/basic_m1/model/basic_model.tflite
📦 Model size: 0.12 MB


2025-09-02 18:02:19.231267: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-09-02 18:02:19.231283: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-09-02 18:02:19.231437: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/tmp8jjt7rky
2025-09-02 18:02:19.232839: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-09-02 18:02:19.232843: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/tmp8jjt7rky
2025-09-02 18:02:19.236273: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2025-09-02 18:02:19.285890: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/tmp8jjt7rky
2025-09-

In [28]:
# Test TFLite model with test_data files
if tflite_path:
    print("🧪 Testing TFLite model with test_data files...")
    
    # Load TFLite model
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    # Get input and output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    print(f"Input shape: {input_details[0]['shape']}")
    print(f"Input type: {input_details[0]['dtype']}")
    print(f"Output shape: {output_details[0]['shape']}")
    print(f"Output type: {output_details[0]['dtype']}")
    
    # Test with all files in test_data
    test_data_dir = "../../test_data"
    test_files = []
    
    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        test_files.extend(glob(os.path.join(test_data_dir, ext)))
    
    print(f"\n📁 Found {len(test_files)} test files")
    print("=" * 60)
    
    results = []
    
    for img_path in sorted(test_files):
        filename = os.path.basename(img_path)
        
        # Extract expected label from filename prefix
        expected_label = filename.split('_')[0] if '_' in filename else filename.split('.')[0]
        
        # Load and preprocess image
        try:
            bgr = cv2.imread(img_path)
            if bgr is None:
                print(f"❌ {filename}: Could not load image")
                continue
                
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb = cv2.resize(rgb, config["image_size"])
            input_data = (rgb.astype(np.float32) / 255.0)[None, ...]
            
            # Run inference
            interpreter.set_tensor(input_details[0]['index'], input_data)
            interpreter.invoke()
            
            # Get predictions
            predictions = interpreter.get_tensor(output_details[0]['index'])[0]
            pred_idx = int(np.argmax(predictions))
            pred_class = config["class_names"][pred_idx]
            confidence = float(np.max(predictions))
            
            # Check if prediction matches expected
            is_correct = pred_class.lower() == expected_label.lower()
            status = "✅" if is_correct else "❌"
            
            print(f"{status} {filename:30} | Expected: {expected_label:8} | Predicted: {pred_class:8} | Conf: {confidence:.3f}")
            
            results.append({
                'filename': filename,
                'expected': expected_label,
                'predicted': pred_class,
                'confidence': confidence,
                'correct': is_correct
            })
            
        except Exception as e:
            print(f"❌ {filename}: Error - {str(e)}")
    
    # Summary
    correct_count = sum(1 for r in results if r['correct'])
    total_count = len(results)
    accuracy = correct_count / total_count if total_count > 0 else 0
    
    print("=" * 60)
    print(f"📊 Summary: {correct_count}/{total_count} correct ({accuracy:.1%})")
    
    # Show per-class breakdown
    print("\n📈 Per-class results:")
    for class_name in config["class_names"]:
        class_results = [r for r in results if r['expected'].lower() == class_name.lower()]
        if class_results:
            class_correct = sum(1 for r in class_results if r['correct'])
            class_total = len(class_results)
            print(f"  {class_name:8}: {class_correct}/{class_total} ({class_correct/class_total:.1%})")
    
else:
    print("❌ Cannot test TFLite model - conversion failed")


🧪 Testing TFLite model with test_data files...
Input shape: [  1 224 224   3]
Input type: <class 'numpy.float32'>
Output shape: [1 5]
Output type: <class 'numpy.float32'>

📁 Found 10 test files
❌ benign_27.jpg                  | Expected: benign   | Predicted: OCA      | Conf: 0.511
❌ benign_42.jpg                  | Expected: benign   | Predicted: OCA      | Conf: 0.453
❌ healthy_C-4-6-2.jpg            | Expected: healthy  | Predicted: Unknown  | Conf: 0.546
❌ healthy_C-5-6-1.jpg            | Expected: healthy  | Predicted: Benign   | Conf: 0.360
✅ oca_26.jpg                     | Expected: oca      | Predicted: OCA      | Conf: 0.458
❌ oca_27.jpg                     | Expected: oca      | Predicted: Benign   | Conf: 0.278
❌ opmd_K-005-03.jpg              | Expected: opmd     | Predicted: Benign   | Conf: 0.339
❌ opmd_K-006-01.jpg              | Expected: opmd     | Predicted: Benign   | Conf: 0.434
✅ unknown_000000002431.jpg       | Expected: unknown  | Predicted: Unknown  | Conf: 0.

In [29]:
# Save detailed results to CSV
if 'results' in locals() and results:
    import pandas as pd
    
    results_df = pd.DataFrame(results)
    results_csv_path = os.path.join(config["output_root"], "test_data_results.csv")
    results_df.to_csv(results_csv_path, index=False)
    
    print(f"💾 Detailed results saved to: {results_csv_path}")
    
    # Show the DataFrame
    print("\n📋 Results DataFrame:")
    print(results_df)


💾 Detailed results saved to: ./ml/training/results/basic_m1/test_data_results.csv

📋 Results DataFrame:
                   filename expected predicted  confidence  correct
0             benign_27.jpg   benign       OCA    0.511176    False
1             benign_42.jpg   benign       OCA    0.452917    False
2       healthy_C-4-6-2.jpg  healthy   Unknown    0.546363    False
3       healthy_C-5-6-1.jpg  healthy    Benign    0.359513    False
4                oca_26.jpg      oca       OCA    0.457828     True
5                oca_27.jpg      oca    Benign    0.277514    False
6         opmd_K-005-03.jpg     opmd    Benign    0.339017    False
7         opmd_K-006-01.jpg     opmd    Benign    0.433588    False
8  unknown_000000002431.jpg  unknown   Unknown    0.410818     True
9  unknown_000000002592.jpg  unknown   Unknown    0.944325     True


In [30]:
# Create labels.txt file for Flutter app
print("📝 Creating labels.txt file for Flutter app...")

labels_txt_path = os.path.join(save_dir, "labels.txt")
with open(labels_txt_path, 'w') as f:
    for class_name in config["class_names"]:
        f.write(f"{class_name}\n")

print(f"✅ Labels file saved to: {labels_txt_path}")
print("📋 Contents:")
with open(labels_txt_path, 'r') as f:
    for i, line in enumerate(f):
        print(f"  {i}: {line.strip()}")

# Also copy to assets folder for Flutter app
assets_labels_path = "../../assets/labels.txt"
try:
    import shutil
    shutil.copy2(labels_txt_path, assets_labels_path)
    print(f"✅ Also copied to: {assets_labels_path}")
except Exception as e:
    print(f"⚠️ Could not copy to assets folder: {e}")


📝 Creating labels.txt file for Flutter app...
✅ Labels file saved to: ./ml/training/results/basic_m1/model/labels.txt
📋 Contents:
  0: Benign
  1: Healthy
  2: OCA
  3: OPMD
  4: Unknown
⚠️ Could not copy to assets folder: [Errno 2] No such file or directory: '../../assets/labels.txt'


In [31]:
# Advanced techniques for handling class imbalance
print("🔧 Advanced Class Imbalance Techniques")
print("=" * 50)

print("\n1. 📊 Current approach: Class Weights")
print("   ✅ Automatically calculated based on class frequency")
print("   ✅ Applied during training to balance loss")
print("   ✅ Works well for moderate imbalance")

print("\n2. 🎯 Alternative approaches you can try:")
print("   • Focal Loss: Focuses on hard-to-classify examples")
print("   • SMOTE: Synthetic data generation for minority classes")
print("   • Cost-sensitive learning: Different costs for different misclassifications")
print("   • Ensemble methods: Train multiple models with different sampling")

print("\n3. 📈 Monitoring class performance:")
print("   • Per-class precision, recall, F1-score")
print("   • Confusion matrix analysis")
print("   • ROC curves for each class")
print("   • Precision-Recall curves")

print("\n4. 🎛️ Hyperparameter tuning for imbalanced data:")
print("   • Learning rate: Lower for better convergence")
print("   • Batch size: Smaller batches for better gradient estimates")
print("   • Epochs: More epochs needed for minority classes")
print("   • Validation: Use stratified splits")

# Example of how to implement Focal Loss (advanced)
print("\n5. 🔬 Focal Loss implementation (for reference):")
print("""
def focal_loss(alpha=0.25, gamma=2.0):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = alpha_t * tf.pow((1 - p_t), gamma)
        
        focal_loss = -focal_weight * tf.math.log(p_t)
        return tf.reduce_mean(focal_loss)
    return focal_loss_fixed
""")

print("\n💡 Recommendation: Start with class weights, then experiment with Focal Loss if needed")


🔧 Advanced Class Imbalance Techniques

1. 📊 Current approach: Class Weights
   ✅ Automatically calculated based on class frequency
   ✅ Applied during training to balance loss
   ✅ Works well for moderate imbalance

2. 🎯 Alternative approaches you can try:
   • Focal Loss: Focuses on hard-to-classify examples
   • SMOTE: Synthetic data generation for minority classes
   • Cost-sensitive learning: Different costs for different misclassifications
   • Ensemble methods: Train multiple models with different sampling

3. 📈 Monitoring class performance:
   • Per-class precision, recall, F1-score
   • Confusion matrix analysis
   • ROC curves for each class
   • Precision-Recall curves

4. 🎛️ Hyperparameter tuning for imbalanced data:
   • Learning rate: Lower for better convergence
   • Batch size: Smaller batches for better gradient estimates
   • Epochs: More epochs needed for minority classes
   • Validation: Use stratified splits

5. 🔬 Focal Loss implementation (for reference):

def foca

In [33]:
reloaded = keras.models.load_model(model_path)
print("Reloaded.")

# Pick a few images from your test directory
from glob import glob
import cv2

# Try to locate test images; you can manually set a path if needed
candidate_dirs = [
    os.path.join(config["data_dir"], "test", c) for c in (config["class_names"]) if os.path.isdir(os.path.join(config["data_dir"], "test", c))
]
img_paths = []
for d in candidate_dirs:
    img_paths.extend(glob(os.path.join(d, "*.jpg")))

if not img_paths:
    # Fallback: use repo's ml/test_data images
    fallback_dir = "./ml/test_data"
    img_paths = glob(os.path.join(fallback_dir, "*.jpg"))

print("Using", len(img_paths), "test images for quick inference demo (first 4 shown).")

def load_image_for_model(p, size):
    bgr = cv2.imread(p)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, size)
    arr = (rgb.astype(np.float32) / 255.0)[None, ...]
    return arr

for p in img_paths[:4]:
    x = load_image_for_model(p, config["image_size"])
    probs = reloaded.predict(x, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_class = config["class_names"][pred_idx] if pred_idx < len(config["class_names"]) else str(pred_idx)
    print(os.path.basename(p), "->", pred_class, "(prob=", float(np.max(probs)), ")")


Reloaded.
Using 533 test images for quick inference demo (first 4 shown).
R-103-01.jpg -> OPMD (prob= 0.28505948185920715 )
S-124-01.jpg -> Benign (prob= 0.2938510477542877 )
S-97-01.jpg -> OCA (prob= 0.6813176870346069 )
88.jpg -> OCA (prob= 0.7898755073547363 )
